In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# ---- Step 1: Load data ----
df = pd.read_csv("../data/credit_risk_dataset.csv")
print("Shape:", df.shape)
print(df.isnull().sum())

Shape: (32581, 12)
person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64


In [3]:
# ---- Step 2: EDA plots ----
sns.countplot(x="loan_status", data=df)
plt.savefig("eda_loan_status.png"); plt.close()

sns.histplot(df["person_age"], bins=30)
plt.savefig("eda_age_hist.png"); plt.close()

sns.boxplot(x=df["person_age"])
plt.savefig("eda_age_box.png"); plt.close()

plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm")
plt.tight_layout()
plt.savefig("eda_corr_heatmap.png"); plt.close()

In [4]:
# ---- Step 3: Handle missing values ----
print("\nMissing before:\n", df.isnull().sum())

df["person_emp_length"] = df["person_emp_length"].fillna(df["person_emp_length"].median())
df["loan_int_rate"] = df["loan_int_rate"].fillna(df["loan_int_rate"].median())

print("\nMissing after:\n", df.isnull().sum())


Missing before:
 person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

Missing after:
 person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_status                   0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
dtype: int64


In [5]:
# ---- Step 4: Remove invalid rows (age/emp_length outliers) ----
df = df[df["person_age"] > 18]
df = df[df["person_age"] < 100]
df = df[df["person_emp_length"] < 80]

print("\nShape after cleaning:", df.shape)


Shape after cleaning: (32574, 12)


In [6]:
# ---- Step 5: Feature engineering ----
# Income-to-loan ratio context (analogous to DebtIncomeRatio)
df["LoanToIncomeAmount"] = df["loan_amnt"] / df["person_income"]

# Encode categorical columns (one-hot, analogous to preparing all-numeric x)
cat_cols = ["person_home_ownership", "loan_intent", "loan_grade", "cb_person_default_on_file"]
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print("\nEncoded shape:", df_encoded.shape)


Encoded shape: (32574, 24)


In [7]:
# ---- Step 6: Target and Features ----
x = df_encoded.drop("loan_status", axis=1)
y = df_encoded["loan_status"]

In [8]:
# ---- Step 7: Train/test split ----
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

In [9]:
# ---- Step 8: Scaling ----
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

import joblib
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [10]:
# ---- Step 9: Logistic Regression ----
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(class_weight="balanced", max_iter=1000)
lr.fit(x_train_scaled, y_train)

y_pred_lr = lr.predict(x_test_scaled)
y_prob_lr = lr.predict_proba(x_test_scaled)[:, 1]

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
print("\n--- Logistic Regression ---")
print(classification_report(y_test, y_pred_lr))
print(confusion_matrix(y_test, y_pred_lr))

auc_lr = roc_auc_score(y_test, y_prob_lr)
results = {"Logistic regression": auc_lr}
print("Results:", results)


--- Logistic Regression ---
              precision    recall  f1-score   support

           0       0.93      0.83      0.88      5094
           1       0.56      0.76      0.64      1421

    accuracy                           0.82      6515
   macro avg       0.74      0.80      0.76      6515
weighted avg       0.84      0.82      0.82      6515

[[4228  866]
 [ 339 1082]]
Results: {'Logistic regression': 0.8695698904231689}


In [11]:
# ---- Step 10: XGBoost ----
from xgboost import XGBClassifier

neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos

xgb = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    eval_metric="auc",
    random_state=42
)
xgb.fit(x_train, y_train)

y_pred_xgb = xgb.predict(x_test)
y_prob_xgb = xgb.predict_proba(x_test)[:, 1]

auc_xgb = roc_auc_score(y_test, y_prob_xgb)
results["XGBoost"] = auc_xgb
print("\nXGBoost AUC:", auc_xgb)

print("\n--- XGBoost ---")
print(classification_report(y_test, y_pred_xgb))
print(confusion_matrix(y_test, y_pred_xgb))
print("\nResults:", results)


XGBoost AUC: 0.9278660410185764

--- XGBoost ---
              precision    recall  f1-score   support

           0       0.94      0.93      0.93      5094
           1       0.75      0.78      0.76      1421

    accuracy                           0.90      6515
   macro avg       0.84      0.85      0.85      6515
weighted avg       0.90      0.90      0.90      6515

[[4729  365]
 [ 319 1102]]

Results: {'Logistic regression': 0.8695698904231689, 'XGBoost': 0.9278660410185764}


In [12]:
# ---- Step 11: Threshold tuning ----
from sklearn.metrics import precision_score, recall_score, f1_score
thresholds = [0.2, 0.3, 0.4, 0.5]
result = []
for threshold in thresholds:
    y_pred = (y_prob_xgb >= threshold).astype(int)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    result.append([threshold, precision, recall, f1])

threshold_df = pd.DataFrame(result, columns=["Threshold", "Precision", "Recall", "F1 Score"])
print("\nThreshold results:\n", threshold_df)

y_pred_02 = (y_prob_xgb >= 0.2).astype(int)
print("\nConfusion matrix @ 0.2:\n", confusion_matrix(y_test, y_pred_02))


Threshold results:
    Threshold  Precision    Recall  F1 Score
0        0.2   0.377703  0.946517  0.539944
1        0.3   0.507076  0.882477  0.644068
2        0.4   0.627620  0.821956  0.711761
3        0.5   0.751193  0.775510  0.763158

Confusion matrix @ 0.2:
 [[2878 2216]
 [  76 1345]]


In [13]:
# ---- Step 12: Feature importance ----
importance_df = pd.DataFrame({
    "Feature": x_train.columns,
    "Importance": xgb.feature_importances_
}).sort_values(by="Importance", ascending=False)
print("\nFeature importances:\n", importance_df)

from xgboost import plot_importance
plot_importance(xgb, max_num_features=10)
plt.tight_layout()
plt.savefig("feature_importance.png")
plt.close()


Feature importances:
                         Feature  Importance
7            LoanToIncomeAmount    0.168958
17                 loan_grade_C    0.116368
4                 loan_int_rate    0.105528
10   person_home_ownership_RENT    0.099828
18                 loan_grade_D    0.081502
1                 person_income    0.057297
9     person_home_ownership_OWN    0.054550
22  cb_person_default_on_file_Y    0.051603
12  loan_intent_HOMEIMPROVEMENT    0.039657
15          loan_intent_VENTURE    0.031892
16                 loan_grade_B    0.028357
11        loan_intent_EDUCATION    0.021680
19                 loan_grade_E    0.020805
13          loan_intent_MEDICAL    0.020530
2             person_emp_length    0.016986
20                 loan_grade_F    0.016749
0                    person_age    0.013050
21                 loan_grade_G    0.012394
14         loan_intent_PERSONAL    0.011990
5           loan_percent_income    0.010840
8   person_home_ownership_OTHER    0.008730
3        

In [14]:
import shap
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(x_test)

shap.summary_plot(shap_values, x_test, show=False)
plt.tight_layout()
plt.savefig("shap_summary.png", bbox_inches="tight")
plt.close()

In [15]:
# ---- Step 14: Save artifacts ----
joblib.dump(xgb, "loan_default_xgb.pkl")
joblib.dump(scaler, "scaler.pkl")
BestThreshold = 0.2

features_name = x.columns.tolist()
print("\nFeatures:", features_name)

import os
print("\nOutput files:", os.listdir())


Features: ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'LoanToIncomeAmount', 'person_home_ownership_OTHER', 'person_home_ownership_OWN', 'person_home_ownership_RENT', 'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE', 'loan_grade_B', 'loan_grade_C', 'loan_grade_D', 'loan_grade_E', 'loan_grade_F', 'loan_grade_G', 'cb_person_default_on_file_Y']

Output files: ['eda_age_box.png', 'eda_age_hist.png', 'eda_corr_heatmap.png', 'eda_loan_status.png', 'feature_importance.png', 'loan_default_xgb.pkl', 'model.ipynb', 'scaler.pkl', 'shap_summary.png']
